# 1.3 — Brightway matching

Searches the selected Brightway project for a real technosphere/biosphere node
for every included flow, then picks one using `SEARCH_CANDIDATE_INDEX` from
[ai_lca_config.py](ai_lca_config.py) — the same `candidate_index` pattern
`dashboard_config.GRID_CANDIDATE_INDEX` uses: `0` is the top search hit, and
you only need to list a flow there if you want something other than the top
match. `SEARCH_QUERY_OVERRIDE` lets you replace the search text entirely for a
flow before it's re-searched.

Geography is a soft ranking hint only — it never filters candidates out.
Emissions are searched in the biosphere database; everything else in the
technosphere database. A flow with an explicit `linked_process_id` (a
foreground-to-foreground link) needs no Brightway mapping at all.

In [ ]:
import pandas as pd

import ai_lca_config as cfg
cfg.print_config()

import bw2data as bd

from ai_lca.brightway_search import list_biosphere_databases, list_databases
from ai_lca.geography import ecoinvent_location_hints
from ai_lca.notebook_helpers import build_mapping_row, load_extraction, print_candidates, run_output_dir, search_flow_candidates

run_dir = run_output_dir(cfg.OUTPUT_DIR, cfg.RUN_LABEL)
reviewed_path = run_dir / "1_1_extraction_reviewed.json"
inventory_path = run_dir / "1_2_inventory_reviewed.csv"
for p in (reviewed_path, inventory_path):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found — run the earlier 1.x notebooks first.")

extraction = load_extraction(reviewed_path)
inventory_df = pd.read_csv(inventory_path)
included_df = inventory_df[inventory_df["include"] == True].reset_index(drop=True)  # noqa: E712
print(f"{len(included_df)} of {len(inventory_df)} flow(s) included for matching.")

## Select the Brightway project and databases

In [ ]:
bd.projects.set_current(cfg.BRIGHTWAY_PROJECT)
all_dbs = list_databases(cfg.BRIGHTWAY_PROJECT)
biosphere_dbs = list_biosphere_databases(cfg.BRIGHTWAY_PROJECT)
technosphere_dbs = [d for d in all_dbs if d not in biosphere_dbs]

technosphere_database = cfg.TECHNOSPHERE_DATABASE or next(
    (d for d in technosphere_dbs if "ecoinvent" in d.lower()), technosphere_dbs[0] if technosphere_dbs else None,
)
biosphere_database = biosphere_dbs[0] if biosphere_dbs else None

print("Brightway project:      ", cfg.BRIGHTWAY_PROJECT)
print("Technosphere database:  ", technosphere_database)
print("Biosphere database:     ", biosphere_database or "(none found — emission flows will be blocked)")

location_hints = ecoinvent_location_hints(extraction.study_context.operational_geography)
print("Geography hints:        ", location_hints or "(none)")

## Search candidates for every included flow

In [ ]:
candidates_by_flow: dict[int, list[dict]] = {}

for _, row in included_df.iterrows():
    flow_id = int(row["flow_id"])
    search_row = dict(row)
    override_query = cfg.SEARCH_QUERY_OVERRIDE.get(flow_id)
    if override_query:
        search_row["name"] = override_query
        search_row["background_process_hint"] = ""

    candidates = search_flow_candidates(
        search_row,
        project_name=cfg.BRIGHTWAY_PROJECT,
        database_name=technosphere_database,
        biosphere_database=biosphere_database,
        candidate_limit=cfg.CANDIDATE_LIMIT,
        location_hints=location_hints,
    )
    candidates_by_flow[flow_id] = candidates
    chosen_index = cfg.SEARCH_CANDIDATE_INDEX.get(flow_id, 0)
    print_candidates(row, candidates, chosen_index)

## Build the mapping table from your selections

Edit `SEARCH_CANDIDATE_INDEX` / `SEARCH_QUERY_OVERRIDE` in `ai_lca_config.py`
and re-run this notebook to change a choice.

In [ ]:
mapping_rows = []
unmapped = []
for _, row in included_df.iterrows():
    flow_id = int(row["flow_id"])
    candidates = candidates_by_flow[flow_id]
    chosen_index = cfg.SEARCH_CANDIDATE_INDEX.get(flow_id, 0)
    mapping_row = build_mapping_row(row, candidates, chosen_index)
    if mapping_row is not None:
        mapping_rows.append(mapping_row)
    elif not (candidates and "foreground_link" in candidates[0]):
        unmapped.append(flow_id)

mapping_df = pd.DataFrame(mapping_rows)
print(f"{len(mapping_rows)} flow(s) mapped to a real Brightway node.")
if unmapped:
    print(f"{len(unmapped)} flow(s) still unmapped (no candidates / index out of range): {unmapped}")
mapping_df

## Save mapping

In [ ]:
out_path = run_dir / "1_3_mapping.csv"
mapping_df.to_csv(out_path, index=False)
print("Saved mapping to:", out_path)
print()
print("Next: run 1.4.paper_write_foreground.ipynb.")